In [1]:
import json
import re
import time

import pandas as pd
import numpy as np

from tqdm.auto import tqdm

from google import genai
from google.genai import types
import os
import getpass

In [2]:
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:

%pwd
%cd /content/drive/MyDrive/EMP/Notebook

/content/drive/MyDrive/EMP/Notebook


In [5]:

%pwd

'/content/drive/MyDrive/EMP/Notebook'

In [6]:
sent_df = pd.read_csv(CSV_PATH + "Module 1 sentence.csv")

In [7]:
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

for model in client.models.list():
    print(model.name)
    print("supported actions:", getattr(model, "supported_actions", None))
    print("-" * 80)

Enter Gemini API key: ··········
models/gemini-2.5-flash
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.5-pro
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash-001
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash-lite-001
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
-------

In [8]:
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [9]:
LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

LABEL_DEFINITIONS = """
You are labeling sentences for a qualitative research project.

Your task is to assign binary labels for four forms of oppression.

Important:
Every sentence must receive at least one label.
Do not return all 0s.
If the sentence is ambiguous, choose the closest fitting label.
If more than one label clearly applies, you may assign multiple 1s.
If only one label is strongest, assign only that label as 1.

Use only the information inside the sentence.
Do not create a fifth label.
Do not use "none".

Labels:

1. ideological
Use 1 when the sentence is mainly about beliefs, stereotypes, assumptions, norms, dominant ideas, cultural expectations, or values that shape inequality.

Choose ideological when the sentence focuses on:
- what society believes
- what people are expected to believe
- stereotypes about groups
- cultural narratives
- assumptions about ability, language, race, gender, class, religion, disability, or identity

Examples:
- People assume English speakers are smarter.
- Society treats lighter skin as more beautiful.
- There is a belief that men are naturally better leaders.

2. institutionalized
Use 1 when the sentence is mainly about systems, institutions, policies, schools, workplaces, religion, government, laws, programs, formal rules, access to resources, or structural barriers.

Choose institutionalized when the sentence focuses on:
- school rules or practices
- workplace structures
- government/law/policy
- access to education, healthcare, money, jobs, housing, or services
- institutional power

Examples:
- School policies made it harder for immigrant students to get help.
- The workplace did not provide disability accommodations.
- Government rules limited access to healthcare.

3. interpersonal
Use 1 when the sentence is mainly about person-to-person treatment, exclusion, judgment, insults, bullying, microaggressions, discrimination, family pressure, peer treatment, or conflict with others.

Choose interpersonal when the sentence focuses on:
- what someone said
- how people treated the speaker
- family, peers, classmates, teachers, coworkers, or strangers judging/excluding someone
- direct interaction or mistreatment

Examples:
- My classmates mocked my accent.
- People treated me differently because of my background.
- My family judged me for not following their expectations.

4. internalized
Use 1 when the sentence is mainly about shame, self-doubt, hiding identity, discomfort with oneself, feeling lesser, blaming oneself, or accepting negative beliefs about oneself.

Choose internalized when the sentence focuses on:
- "I felt ashamed"
- "I felt embarrassed"
- "I hid who I was"
- "I thought something was wrong with me"
- feeling inferior because of identity or background

Examples:
- I felt embarrassed speaking my language.
- I tried to hide my culture to fit in.
- I started believing I was not good enough.

Tie-breaking rules:
- If the sentence mentions a system, policy, school, workplace, law, or formal structure, prefer institutionalized.
- If the sentence mentions direct treatment by people, prefer interpersonal.
- If the sentence focuses on the speaker's shame, self-doubt, or hiding identity, prefer internalized.
- If the sentence focuses on stereotypes, beliefs, norms, or assumptions, prefer ideological.
- If the sentence is very general, choose the label that best explains the source of oppression.
"""

In [10]:
def build_prompt(sentence):
    return f"""
{LABEL_DEFINITIONS}

Classify the sentence using only these four binary labels:

- ideological
- institutionalized
- interpersonal
- internalized

Rules:
- At least one label must be 1.
- Never return all 0s.
- Return only valid JSON.
- Do not include explanations.
- Do not include scores.
- Do not include "none".
- Do not include extra keys.

Exact output format:

{{
  "ideological": 0,
  "institutionalized": 0,
  "interpersonal": 0,
  "internalized": 0
}}

Sentence:
\"\"\"{sentence}\"\"\"
"""


def safe_json_parse(text):
    if not text:
        return None

    text = text.strip()

    # Remove possible markdown fences
    text = re.sub(r"^```json", "", text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return None

    return None


def label_sentence_with_gemini(sentence, max_retries=3, sleep_seconds=2):
    prompt = build_prompt(sentence)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json"
                )
            )

            parsed = safe_json_parse(response.text)

            if parsed is None:
                raise ValueError(f"Could not parse JSON: {response.text}")

            # Return only the four labels, nothing else
            result = {
                label: int(parsed.get(label, 0))
                for label in LABELS
            }

            # Force labels to be only 0 or 1
            result = {
                label: 1 if result[label] == 1 else 0
                for label in LABELS
            }

            return result

        except Exception as e:
            if attempt == max_retries - 1:
                # Still return only the four labels
                return {
                    label: 0
                    for label in LABELS
                }

            time.sleep(sleep_seconds * (attempt + 1))

In [11]:
sample_df = sent_df.copy()

In [12]:
labeled_rows = []

for _,row in tqdm(sample_df.iterrows(), total = len(sample_df)):
    sentence = row["sentence"]
    labels = label_sentence_with_gemini(sentence)

    combined = row.to_dict()
    combined.update(labels)

    labeled_rows.append(combined)

gemini_df = pd.DataFrame(labeled_rows)

gemini_df

  0%|          | 0/154 [00:00<?, ?it/s]

,source_id,campus,sentence_index,sentence_id,sentence,ideological,institutionalized,interpersonal,internalized
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...,0,0,0,1
1,1,Chico,1,1_1,I think it is important to know how each indiv...,1,0,0,0
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...,1,0,0,0
3,1,Chico,3,1_3,Since we all have so many different identities...,1,0,0,0
4,1,Chico,4,1_4,I want to continue listening to my students in...,0,1,0,0
...,...,...,...,...,...,...,...,...,...
149,22,San Francisco,0,22_0,"Age: 56 years, my son calls me old a number of...",0,0,1,0
150,23,San Francisco,0,23_0,"Status: Lecturer Faculty, Instructor",0,1,0,0
151,24,San Francisco,0,24_0,Gender: Male,1,0,0,0
152,25,San Francisco,0,25_0,Disability: Wearing eye glasses,1,0,0,0


In [13]:
all_col = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]
gemini_df["primary_leaning"] = gemini_df[all_col].idxmax(axis=1)
gemini_df

,source_id,campus,sentence_index,sentence_id,sentence,ideological,institutionalized,interpersonal,internalized,primary_leaning
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...,0,0,0,1,internalized
1,1,Chico,1,1_1,I think it is important to know how each indiv...,1,0,0,0,ideological
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...,1,0,0,0,ideological
3,1,Chico,3,1_3,Since we all have so many different identities...,1,0,0,0,ideological
4,1,Chico,4,1_4,I want to continue listening to my students in...,0,1,0,0,institutionalized
...,...,...,...,...,...,...,...,...,...,...
149,22,San Francisco,0,22_0,"Age: 56 years, my son calls me old a number of...",0,0,1,0,interpersonal
150,23,San Francisco,0,23_0,"Status: Lecturer Faculty, Instructor",0,1,0,0,institutionalized
151,24,San Francisco,0,24_0,Gender: Male,1,0,0,0,ideological
152,25,San Francisco,0,25_0,Disability: Wearing eye glasses,1,0,0,0,ideological


In [14]:
gemini_df.to_csv(CSV_PATH + "Module 1 Sentences Gemini.csv", index = False)
print(f"Saved Gemini weka labels to: {CSV_PATH}")

Saved Gemini weka labels to: ../Data/
